# 3 2 count analysis weekly

Peer-review copy. Inputs are pseudonymized processed data; see README.md and DATA_DICTIONARY.md. Original preparation/collection steps are outside this package.

In [ ]:
from review_setup import prepare_workspace
prepare_workspace()
from review_setup import initialize_stata
initialize_stata()


In [ ]:
import pandas as pd
from utils import get_source_corrections, StataDiD

source_corrections = get_source_corrections(window_days=28, post_correction_free=True).query('prior_corrections==0').reset_index(drop=True)
ts_posts = pd.read_parquet('data/ts_posts_weekly.parquet.gzip').reset_index(drop=True)
ts_posts = ts_posts.merge(source_corrections, on='tweet_id', how='inner')
print(ts_posts.shape)
ts_posts.head()

In [ ]:
stata_did = StataDiD(ts_posts, 'treated', 'tweet_id', 'weeks_from_display', -4, 4)

### Original post weekly [-4, 4]

In [ ]:
stata_did.multi_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
from utils import get_coefs
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=1, unit_group=True)

#### HonestDiD for every post-treatment weeks

In [ ]:
from pystata import stata
stata.run('matrix list e(b)')

In [ ]:
ylims = (-0.6, 0.7)
stata.run(f"local plotopts xtitle(Mbar, size(large)) ytitle(95% Robust CI, size(large)) yscale(r({ylims[0]} {ylims[1]})) ylabel({ylims[0]}(0.2){ylims[1]},labsize(large)) xlabel(,labsize(large)) ciopts(lwidth(thick)) xsize(4) ysize(3)")

for i in range(1,5):
    print(f"Post-treatment period {i}: \n")
    vec = [str(1) if e==i else str(0) for e in range(1,5)]
    vec_str = ' \\ '.join(vec)
    stata.run(f"matrix l_vec = {vec_str}")
    stata.run("honestdid, l_vec(l_vec) pre(9/11) post(12/15) mvec(0.5(0.5)2) alpha(0.05) grid_lb(-3) grid_ub(3)")
    stata.run("honestdid, cached coefplot `plotopts'")
    stata.run(f"graph export outputs/figures/honest_did_posts_weekly_{i}.pdf, replace")

In [ ]:
stata_did.tw_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
from utils import att_plot
ylims = (-0.262, 0.462)
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att, ll_agg_att, hl_agg_att, 'Weekly', 'Weeks from note display', 'Estimated DiD effect', 'did_posts_weekly', ylims=ylims, legend_ncol=1, legend_title='First correction')

### SDiD

In [ ]:
from utils import SEED
stata.run(
    f"""
    sdid tweet_count unitId weeks_from_display treated_after, ///
        vce(bootstrap) cluster(account_cluster) ///
        xline_opts(lcolor(none)) ///
        reps(100) seed({SEED}) graph ///
        g2_opt(xline(0, lcolor(black) lpattern(dash)) xtitle("Weeks from note display") ytitle("Post count") xlabel(-4(1)4) ylabel(25(5)60) title("") graphregion(color(white)))
    """
)

In [ ]:
stata.run(f"graph export outputs/figures/sdid_posts_weekly.pdf, replace")

### NB - mixed effects

In [ ]:
stata_did.multi_did(reg='menbreg', dev='tweet_count', random_unit=True, cluster='account_cluster')

In [ ]:
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='menbreg', start_line=8, end_line=15, stat_type='z', irr_or=1, trans='irr', unit_group=True)

In [ ]:
stata_did.tw_did(reg='menbreg', dev='tweet_count', random_unit=True, cluster='account_cluster')

In [ ]:
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='menbreg', start_line=2, end_line=3, stat_type='z', irr_or=1, trans='irr', unit_group=True)

In [ ]:
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att, ll_agg_att, hl_agg_att, 'Weekly', 'Weeks from note display', 'Estimated DiD effect', 'did_posts_weekly_me', ylims=ylims, legend_ncol=1, legend_title='First correction')

### NB - Fixed effects

In [ ]:
stata_did.multi_did(reg='xtnbreg', dev='tweet_count', random_unit=False, appendix='vce(bootstrap, reps(100))')

In [ ]:
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='xtnbreg', start_line=8, end_line=15, stat_type='z', irr_or=1, trans='irr', unit_group=True)

In [ ]:
stata_did.tw_did(reg='xtnbreg', dev='tweet_count', random_unit=False, appendix='vce(bootstrap, reps(100))')

In [ ]:
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='xtnbreg', start_line=2, end_line=3, stat_type='z', irr_or=1, trans='irr', unit_group=True)

In [ ]:
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att, ll_agg_att, hl_agg_att, 'Weekly', 'Weeks from note display', 'Estimated DiD effect', 'did_posts_weekly_fe', ylims=ylims, legend_ncol=1, legend_title='First correction')

### Fixed effects - Poisson

In [ ]:
stata_did.multi_did(reg='xtpoisson', dev='tweet_count', random_unit=False, cluster=True)

In [ ]:
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='xtpoisson', start_line=7, end_line=14, stat_type='z', irr_or=1, trans='irr', unit_group=True)

In [ ]:
stata_did.tw_did(reg='xtpoisson', dev='tweet_count', random_unit=False, cluster=True)

In [ ]:
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='xtpoisson', start_line=1, end_line=2, stat_type='z', irr_or=1, trans='irr', unit_group=True)

In [ ]:
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att, ll_agg_att, hl_agg_att, 'Weekly', 'Weeks from note display', 'Estimated DiD effect', 'did_posts_weekly_fe_poisson', ylims=ylims, legend_ncol=1, legend_title='First correction')

### Propensity score matching

In [ ]:
# psm based on logistic regression
from sklearn.linear_model import LogisticRegression
from scipy.stats import zscore
from utils import SEED, fit_columns

source_corrections_zscore = source_corrections.dropna(subset=fit_columns).reset_index(drop=True)
print(source_corrections_zscore.shape)
for col in fit_columns:
    if col != 'user_is_verified':
        source_corrections_zscore[col] = zscore(source_corrections_zscore[col])

psm_model = LogisticRegression(C=1e6, random_state=SEED).fit(source_corrections_zscore[fit_columns], source_corrections_zscore['treated'])

In [ ]:
from causalml.match import NearestNeighborMatch

source_corrections_zscore = source_corrections_zscore.assign(propensity_score=psm_model.predict_proba(source_corrections_zscore[fit_columns])[:,1]).reset_index(drop=True)
psm = NearestNeighborMatch(replace=False, ratio=1, random_state=SEED, caliper=0.15)

# control group
control_matched = psm.match(data=source_corrections_zscore, treatment_col='treated', score_cols=['propensity_score']).reset_index(drop=True)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from utils import FONT_SIZE, moderate_cyan, moderate_blue
plt.style.use("default")
plt.rcParams['font.size'] = FONT_SIZE

## visualization before psm
plt.subplots(layout="constrained")
sns.histplot(source_corrections_zscore['propensity_score'][source_corrections_zscore['treated']==0], label='No', element="step", color=moderate_cyan)
sns.histplot(source_corrections_zscore['propensity_score'][source_corrections_zscore['treated']==1], label='Yes', element="step", color=moderate_blue)
plt.xlabel('Propensity score')
plt.ylabel('Count')
plt.legend(title='Treated', loc='upper right')
plt.savefig('outputs/figures/propensity_score_correction.pdf')

In [ ]:
plt.subplots(layout="constrained")
sns.histplot(control_matched.query('treated==0')['propensity_score'], label='No', element="step", color=moderate_cyan)
sns.histplot(control_matched.query('treated==1')['propensity_score'], label='Yes', element="step", color=moderate_blue)
plt.xlabel('Propensity score')
plt.ylabel('Count')
plt.legend(title='Treated')
plt.savefig('outputs/figures/propensity_score_correction_matched.pdf')

In [ ]:
from scipy.stats import ttest_ind
def get_tresult(d1, d2):
    tresult = ttest_ind(d1, d2)
    p_text = '< 0.001' if tresult.pvalue < 0.001 else f'= {tresult.pvalue:.3f}'
    print(f't = {tresult.statistic:.3f} (df = {int(tresult.df):,}), p {p_text}')
get_tresult(source_corrections_zscore['propensity_score'][source_corrections_zscore['treated']==0], source_corrections_zscore['propensity_score'][source_corrections_zscore['treated']==1])

In [ ]:
get_tresult(control_matched.query('treated==0')['propensity_score'], control_matched.query('treated==1')['propensity_score'])

In [ ]:
source_corrections_zscore = source_corrections_zscore.merge(control_matched[['tweet_id']], on='tweet_id', how='inner')
source_corrections_zscore.to_parquet('outputs/data/source_corrections_matched.parquet')
print(source_corrections_zscore.shape)

In [ ]:
from utils import check_balance
balance = check_balance(source_corrections_zscore, fit_columns, tab_export=True, tab_label='covariate_balance_psm')

In [ ]:
balance.round(3)

#### Reload

In [ ]:
ts_posts_psm = ts_posts.merge(source_corrections_zscore[['tweet_id']], on='tweet_id', how='inner')
print(ts_posts_psm.shape)
ts_posts_psm.head()

In [ ]:
stata_did = StataDiD(ts_posts_psm, 'treated', 'tweet_id', 'weeks_from_display', -4, 4)

In [ ]:
stata_did.multi_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
stata_did.tw_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att, ll_agg_att, hl_agg_att, 'Weekly', 'Weeks from note display', 'Estimated DiD effect', 'did_posts_weekly_psm', ylims=ylims, legend_ncol=1, legend_title='First correction')

### Note score control

In [ ]:
# Processed membership from the original CoreModel (v1.1) check:
# retain controls with note_score_control >= 0.35.
source_corrections_note_score = pd.read_parquet('data/source_corrections_note_score.parquet')
print(source_corrections_note_score.shape)
source_corrections_note_score.head()


In [ ]:
ts_posts_note_score = ts_posts.merge(source_corrections_note_score[['tweet_id']], on='tweet_id', how='inner')
print(ts_posts_note_score.shape)
ts_posts_note_score.head()

In [ ]:
stata_did = StataDiD(ts_posts_note_score, 'treated', 'tweet_id', 'weeks_from_display', -4, 4)

In [ ]:
stata_did.multi_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
stata_did.tw_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att, ll_agg_att, hl_agg_att, 'Weekly', 'Weeks from note display', 'Estimated DiD effect', 'did_posts_weekly_note_score', ylims=ylims, legend_ncol=1, legend_title='First correction')